# Export & Deploy World Models

WorldKit models can be exported to ONNX and TorchScript for production
deployment. This notebook shows how to:
1. Train or load a model
2. Export to ONNX
3. Run inference with ONNX Runtime
4. Benchmark PyTorch vs ONNX performance
5. Export to TorchScript

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/06_export_deploy.ipynb)

**Time to run:** ~5 minutes on Colab T4

In [ ]:
!pip install "worldkit[train,export]" -q

## 1. Get a Model

We'll train a small model on synthetic data. You could also
load a pre-trained one with `WorldModel.from_hub()` or `WorldModel.load()`.

In [ ]:
import numpy as np
import h5py
import time
import matplotlib.pyplot as plt
from worldkit import WorldModel

# Quick synthetic data
pixels = np.random.randint(0, 255, (100, 16, 96, 96, 3), dtype=np.uint8)
actions = np.random.randn(100, 16, 2).astype(np.float32)

with h5py.File("export_data.h5", "w") as f:
    f.create_dataset("pixels", data=pixels, compression="gzip")
    f.create_dataset("actions", data=actions, compression="gzip")

model = WorldModel.train(
    data="export_data.h5",
    config="nano",
    epochs=5,
    batch_size=32,
    action_dim=2,
    device="auto",
)

print(f"Model: {model.num_params:,} params, latent_dim={model.latent_dim}")

## 2. Export to ONNX

ONNX is the most portable format — it runs on CPUs, GPUs, mobile devices,
and edge hardware via ONNX Runtime.

In [ ]:
onnx_path = model.export(
    format="onnx",
    output="./export_onnx/",
    optimize=True,
)

print(f"ONNX model saved to: {onnx_path}")

# Check file size
import os
for f in os.listdir(onnx_path):
    fpath = os.path.join(onnx_path, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f}: {size_mb:.1f} MB")

## 3. Run Inference with ONNX Runtime

ONNX Runtime provides optimized inference without requiring PyTorch.

In [ ]:
import onnxruntime as ort
import glob

# Find the ONNX file
onnx_files = glob.glob(os.path.join(str(onnx_path), "*.onnx"))
print(f"ONNX file: {onnx_files[0]}")

# Create session
session = ort.InferenceSession(onnx_files[0])

# Show inputs and outputs
print("\nInputs:")
for inp in session.get_inputs():
    print(f"  {inp.name}: {inp.shape} ({inp.type})")

print("\nOutputs:")
for out in session.get_outputs():
    print(f"  {out.name}: {out.shape} ({out.type})")

In [ ]:
# Run inference with ONNX Runtime
test_frame = np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8)

# Prepare input (match expected format from ONNX model)
# The encoder expects (B, C, H, W) float32
input_tensor = test_frame.astype(np.float32).transpose(2, 0, 1) / 255.0
input_tensor = np.expand_dims(input_tensor, 0)  # Add batch dim

# Run
input_name = session.get_inputs()[0].name
onnx_output = session.run(None, {input_name: input_tensor})

print(f"ONNX output shape: {onnx_output[0].shape}")
print(f"ONNX output norm:  {np.linalg.norm(onnx_output[0]):.4f}")

# Compare with PyTorch output
pytorch_output = model.encode(test_frame)
print(f"\nPyTorch output shape: {pytorch_output.shape}")
print(f"PyTorch output norm:  {np.linalg.norm(pytorch_output):.4f}")

# Check they match
diff = np.abs(onnx_output[0].flatten() - pytorch_output).max()
print(f"\nMax difference: {diff:.6f}")
print(f"Match: {'Yes' if diff < 0.01 else 'No (check precision)'}")

## 4. Benchmark: PyTorch vs ONNX

Let's measure encoding latency for both runtimes.

In [ ]:
n_warmup = 10
n_benchmark = 100

# Generate test frames
test_frames = [np.random.randint(0, 255, (96, 96, 3), dtype=np.uint8) for _ in range(n_benchmark)]

# Prepare ONNX inputs
onnx_inputs = [
    np.expand_dims(f.astype(np.float32).transpose(2, 0, 1) / 255.0, 0)
    for f in test_frames
]

# --- PyTorch benchmark ---
# Warmup
for i in range(n_warmup):
    model.encode(test_frames[i])

pytorch_times = []
for frame in test_frames:
    t0 = time.perf_counter()
    model.encode(frame)
    pytorch_times.append((time.perf_counter() - t0) * 1000)

# --- ONNX benchmark ---
# Warmup
for i in range(n_warmup):
    session.run(None, {input_name: onnx_inputs[i]})

onnx_times = []
for inp in onnx_inputs:
    t0 = time.perf_counter()
    session.run(None, {input_name: inp})
    onnx_times.append((time.perf_counter() - t0) * 1000)

print(f"PyTorch encode: {np.mean(pytorch_times):.2f} +/- {np.std(pytorch_times):.2f} ms")
print(f"ONNX encode:    {np.mean(onnx_times):.2f} +/- {np.std(onnx_times):.2f} ms")
print(f"Speedup:        {np.mean(pytorch_times) / np.mean(onnx_times):.2f}x")

In [ ]:
# Visualize latency distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(pytorch_times, bins=20, alpha=0.7, color="steelblue", label="PyTorch")
axes[0].hist(onnx_times, bins=20, alpha=0.7, color="coral", label="ONNX")
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Encoding latency distribution")
axes[0].legend()

# Bar chart of p50 / p95 / p99
percentiles = [50, 95, 99]
pt_percs = [np.percentile(pytorch_times, p) for p in percentiles]
onnx_percs = [np.percentile(onnx_times, p) for p in percentiles]

x = np.arange(len(percentiles))
w = 0.35
axes[1].bar(x - w/2, pt_percs, w, label="PyTorch", color="steelblue")
axes[1].bar(x + w/2, onnx_percs, w, label="ONNX", color="coral")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"p{p}" for p in percentiles])
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency percentiles")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Export to TorchScript

TorchScript is useful when you need to stay in the PyTorch ecosystem
but want ahead-of-time compilation.

In [ ]:
ts_path = model.export(
    format="torchscript",
    output="./export_torchscript/",
)

print(f"TorchScript model saved to: {ts_path}")

for f in os.listdir(ts_path):
    fpath = os.path.join(ts_path, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f}: {size_mb:.1f} MB")

## 6. Save as .wk for WorldKit Ecosystem

The native `.wk` format preserves everything: weights, config,
metadata, and action space info. Use this for sharing between
WorldKit users.

In [ ]:
model.save(
    "my_model.wk",
    metadata={"description": "Nano model for export demo"},
    action_space={"type": "continuous", "dim": 2, "low": -1.0, "high": 1.0},
)

# Reload and verify
reloaded = WorldModel.load("my_model.wk")
z_orig = model.encode(test_frame)
z_reload = reloaded.encode(test_frame)
print(f"Reload match: {np.allclose(z_orig, z_reload, atol=1e-5)}")

## Deployment Options Summary

| Format | Use case | File size | Inference speed |
|--------|----------|-----------|------------------|
| **.wk** (native) | Sharing, retraining, full WorldKit API | Smallest | PyTorch speed |
| **ONNX** | Production servers, mobile, edge, cross-platform | Medium | Fastest (optimized) |
| **TorchScript** | PyTorch-native deployment, C++ integration | Medium | Fast |
| **TensorRT** | NVIDIA GPUs, maximum throughput | Large | Fastest on GPU |
| **CoreML** | Apple devices (iOS, macOS) | Medium | Fast on Apple Silicon |

For most production deployments, **ONNX** is the recommended format.
Use `.wk` for development and sharing within the WorldKit ecosystem.

## Next Steps

| Notebook | What's next |
|----------|-------------|
| [01 — Quickstart](01_quickstart.ipynb) | Review the basics |
| [05 — Model Comparison](05_model_comparison.ipynb) | Find the right model size before exporting |
| [07 — Plan Robot Actions](07_plan_robot_actions.ipynb) | Use exported models for planning |

**Production resources:**
- **REST API**: `worldkit serve --model my_model.wk --port 8000`
- **Docker**: See the [deployment docs](https://worldkit.readthedocs.io/en/latest/deploy/)
- **Benchmarking**: `worldkit bench quick --model my_model.wk`